# Targeted Description Regeneration

Regenerates descriptions only for rows that failed validation after the
full run: one missing description, and rows containing fabricated
specific distance/time figures (e.g. "1.5 km", "10-minute drive") that
don't exist in the source data. Only categorical proximity labels
(Very Close / Moderately Close / Far) do.

Uses the same model (`qwen3:1.7b`) and the same prompt as the full run,
for consistency.

Objectives:
1. Load the enriched dataset (813 rows, descriptions already generated).
2. Identify rows with a missing description or a fabricated distance/time figure.
3. Regenerate only those rows, retrying if fabrication recurs.
4. Save back into the same file.


In [ ]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
from ollama import chat
import re
from tqdm import tqdm

In [2]:
# ==========================================
# CONFIGURATION
# ==========================================

INPUT_OUTPUT_FILE = "../data/processed/homestays_enriched.csv"

MODEL_NAME = "qwen3:1.7b"

# Retry a flagged row up to this many times if fabrication recurs
MAX_RETRIES = 3

# Matches specific distance/time figures not present in the source data
# (only categorical Very Close / Moderately Close / Far labels exist)
FABRICATION_PATTERN = re.compile(
    r'\d+(?:\.\d+)?\s*(?:km|kilometers?|minutes?|mins?|min walk|hour|hr)',
    re.IGNORECASE,
)

In [3]:
# ==========================================
# LOAD ENRICHED DATASET
# ==========================================

df = pd.read_csv(INPUT_OUTPUT_FILE)
print(f"Total Records: {len(df)}")

Total Records: 813


In [4]:
# ==========================================
# IDENTIFY ROWS NEEDING REGENERATION
# ==========================================

missing_mask = df["description"].isna() | (df["description"].astype(str).str.strip() == "")

has_desc = df["description"].notna()
fabricated_mask = pd.Series(False, index=df.index)
fabricated_mask[has_desc] = df.loc[has_desc, "description"].astype(str).str.contains(FABRICATION_PATTERN, regex=True)

target_mask = missing_mask | fabricated_mask
target_ids = df.loc[target_mask, "homestay_id"].tolist()

print(f"Missing descriptions: {missing_mask.sum()}")
print(f"Fabricated distance/time figures: {fabricated_mask.sum()}")
print(f"Total rows to regenerate: {len(target_ids)}")
print(f"\nhomestay_id list: {target_ids}")

Missing descriptions: 1
Fabricated distance/time figures: 27
Total rows to regenerate: 28

homestay_id list: [59, 92, 121, 165, 200, 217, 219, 241, 243, 300, 302, 314, 315, 322, 379, 415, 550, 635, 689, 732, 763, 901, 913, 948, 1022, 1097, 1109, 1153]


In [5]:
# ==========================================
# WRITING STYLE SELECTION
# ==========================================
# Identical to the full run, kept consistent so regenerated rows use
# the same style-selection logic as the rest of the dataset.

def get_style(row):

    if row.get("mountain_view", 0) == 1:
        return "amenity-focused"

    if row.get("town_proximity", "") == "Very Close":
        return "accessibility-focused"

    if float(row.get("rating", 0) or 0) >= 4.5:
        return "rating-focused"

    return "location-focused"

In [6]:
# ==========================================
# PROMPT CONSTRUCTION
# ==========================================
# Identical to the full run's prompt.

AMENITY_COLUMNS = [
    "wifi", "parking", "breakfast", "mountain_view",
    "room_service", "bonfire_barbeque", "pickup_dropoff_service",
]

def create_prompt(row):

    style = get_style(row)

    amenities = [
        col.replace("_", " ").title()
        for col in AMENITY_COLUMNS
        if row.get(col, 0) == 1
    ]
    amenities_text = ", ".join(amenities) if amenities else "None listed"

    prompt = f"""
Generate a factual homestay description.

Writing Style: {style}

Rules:
- Use ONLY the supplied information.
- Do NOT invent facts.
- Do NOT invent specific distances or travel times (e.g. "1.5 km", "10-minute walk") -- only use the categorical proximity labels supplied below (Very Close / Moderately Close / Far).
- Do NOT mention facilities that are not listed.
- Do NOT mention mountain views unless available.
- You MUST explicitly state the category (Gold or Silver) somewhere in the description -- this is required, not optional.
- Write AT LEAST 55 words and no more than 80 words. Reach this length by including MORE SPECIFIC real details already supplied (the specific village and block names, the exact rating and review count, every listed amenity) -- NOT by adding generic filler phrases like "a wonderful experience" or "nestled in the hills" or "a memorable stay," and NOT by inventing numeric distances.
- Do not include word counts, notes, brackets, or explanations.
- Return ONLY the description.

Style Instructions:

Location-focused:
Start by describing where the homestay is situated.

Amenity-focused:
Highlight the facilities first.

Accessibility-focused:
Focus on proximity to town, Deolo, and Durpin.

Rating-focused:
Naturally mention ratings and reviews.

Data:

Name: {row.get('homestay_name', '')}
Village: {row.get('village', '')}
Block: {row.get('block', '')}
Category: {row.get('category', '')}

Rating: {row.get('rating', '')}
Review Count: {row.get('review_count', '')}

Distance to Town:
{row.get('town_proximity', '')}

Distance to Deolo:
{row.get('deolo_proximity', '')}

Distance to Durpin:
{row.get('durpin_proximity', '')}

Amenities:
{amenities_text}
"""

    return prompt

In [7]:
# ==========================================
# THINKING TRACE CLEANUP
# ==========================================
# Safety net in case a <think>...</think> reasoning trace leaks into the
# response, harmless no-op if none is present.

def strip_thinking(text):
    if "</think>" in text:
        text = text.split("</think>")[-1]
    return text.strip()

In [8]:
# ==========================================
# REGENERATE FLAGGED ROWS (with fabrication retry)
# ==========================================

still_fabricated = []

for hid in tqdm(target_ids):
    row = df[df["homestay_id"] == hid].iloc[0]
    prompt = create_prompt(row)

    final_description = None

    for attempt in range(1, MAX_RETRIES + 1):
        response = chat(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            options={"num_ctx": 4096},
        )
        description = strip_thinking(response["message"]["content"])

        if not FABRICATION_PATTERN.search(description):
            final_description = description
            break

        print(f"\nRow {hid}: attempt {attempt} still fabricated a distance/time figure, retrying...")

    if final_description is None:
        # Kept the last attempt even though it still failed the fabrication
        # check flagged below for manual review rather than silently accepted.
        final_description = description
        still_fabricated.append(hid)

    df.loc[df["homestay_id"] == hid, "description"] = final_description

print(f"\nRegenerated {len(target_ids)} rows.")
if still_fabricated:
    print(f"\nSTILL contain a fabricated figure after {MAX_RETRIES} attempts -- review these manually:")
    print(still_fabricated)
else:
    print("All regenerated rows are clean on the fabrication check.")

  0%|          | 0/28 [00:00<?, ?it/s]

100%|██████████| 28/28 [02:46<00:00,  5.94s/it]


Regenerated 28 rows.
All regenerated rows are clean on the fabrication check.


In [9]:
# ==========================================
# VALIDATION -- FULL DATASET, AFTER REGENERATION
# ==========================================

empty_count = (df["description"].isna() | (df["description"].astype(str).str.strip() == "")).sum()
print(f"Empty descriptions: {empty_count}")

fabricated_count = df["description"].astype(str).str.contains(FABRICATION_PATTERN, regex=True).sum()
print(f"Rows with a fabricated distance/time figure: {fabricated_count}")

word_counts = df["description"].astype(str).str.split().str.len()
print("\nWord count distribution (full dataset):")
print(word_counts.describe())

cat_mentioned = df.apply(lambda r: str(r["category"]).lower() in str(r["description"]).lower(), axis=1)
print(f"\nCategory mentioned: {cat_mentioned.sum()}/{len(df)} ({cat_mentioned.mean()*100:.1f}%)")

Empty descriptions: 0
Rows with a fabricated distance/time figure: 0

Word count distribution (full dataset):
count    813.000000
mean      53.082411
std       10.643777
min       27.000000
25%       46.000000
50%       53.000000
75%       60.000000
max       89.000000
Name: description, dtype: float64

Category mentioned: 540/813 (66.4%)


In [10]:
# ==========================================
# SAVE
# ==========================================

df.to_csv(INPUT_OUTPUT_FILE, index=False)
print(f"Saved {len(df)} records to {INPUT_OUTPUT_FILE}")

Saved 813 records to ../data/processed/homestays_enriched.csv
